# NASA Kepler Objects of Interest - Classification

This notebook prepares the classification track using the NASA Kepler Objects of Interest dataset.

The preprocessing is kept close to the course PDF requirements:

- load and audit the dataset
- check missing values, duplicates, and outliers
- create at least one engineered feature
- split with stratification
- fit imputers/scalers only on training data
- evaluate classifiers with the required classification metrics

## 1. Problem Statement

The task is to predict the official KOI disposition class for a Kepler Object of Interest using tabular measurements from the dataset.

## 2. Dataset Description

Dataset: NASA Kepler Objects of Interest (KOI)

Official source: https://data.nasa.gov/dataset/kepler-objects-of-interest-koi

CSV source used in this notebook: https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=csv

## 3. Imports and Reproducibility

In [ ]:
from pathlib import Path
import io
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
TEST_SIZE = 0.20

sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 100)

## 4. Data Loading

In [ ]:
DATA_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=csv"

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

DATA_PATH = project_root / "data" / "kepler_koi.csv"

if DATA_PATH.exists():
    print(f"Loading local dataset: {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    print("Local dataset not found. Downloading from the Exoplanet Archive...")
    df = pd.read_csv(DATA_URL)
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
    print(f"Saved dataset to: {DATA_PATH}")

## 5. Dataset Audit

In [ ]:
display(df.head())
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

In [ ]:
print("Columns:")
print(df.columns.tolist())

In [ ]:
info_buffer = io.StringIO()
df.info(buf=info_buffer)
print(info_buffer.getvalue())

In [ ]:
audit_table = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2),
    "unique_values": df.nunique(dropna=True),
})

display(audit_table.sort_values("missing_percent", ascending=False))

## 6. Target Definition

In [ ]:
target_column = "koi_disposition"

if target_column not in df.columns:
    raise KeyError(f"Target column not found: {target_column}")

class_counts = df[target_column].value_counts()
class_percentages = df[target_column].value_counts(normalize=True).mul(100).round(2)
target_distribution = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percentages,
})

print(f"Selected classification target: {target_column}")
print(f"Target classes: {sorted(df[target_column].dropna().unique().tolist())}")
display(target_distribution)

`koi_disposition` is used as the target because it directly stores the KOI disposition classes in the dataset.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x=target_column, order=class_counts.index)
plt.title("Target Distribution")
plt.xlabel("KOI disposition")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

> STUDENT OBSERVATION TODO: Comment on the class distribution after running the notebook.

## 7. Exploratory Data Analysis

In [ ]:
eda_features = [
    "koi_period",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "koi_smass",
]

eda_features = [column for column in eda_features if column in df.columns]
display(df[eda_features].describe().T)

In [ ]:
plot_features = eda_features[:9]
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.ravel()

for axis, column in zip(axes, plot_features):
    sns.histplot(df[column], kde=True, ax=axis)
    axis.set_title(f"Distribution of {column}")
    axis.set_xlabel(column)
    axis.set_ylabel("Count")

for axis in axes[len(plot_features):]:
    axis.set_visible(False)

plt.tight_layout()
plt.show()

> STUDENT OBSERVATION TODO: Add a short observation about the distribution plots.

In [ ]:
correlation_features = eda_features[:10]

plt.figure(figsize=(10, 7))
sns.heatmap(df[correlation_features].corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

> STUDENT OBSERVATION TODO: Add a short observation about the correlation heatmap.

In [ ]:
scatter_features = [column for column in ["koi_model_snr", "koi_prad"] if column in df.columns]

for column in scatter_features:
    plt.figure(figsize=(8, 5))
    sns.stripplot(
        data=df,
        x=target_column,
        y=column,
        order=class_counts.index,
        jitter=0.25,
        alpha=0.35,
        size=2,
    )
    plt.title(f"{column} by Target Class")
    plt.xlabel("KOI disposition")
    plt.ylabel(column)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

> STUDENT OBSERVATION TODO: Add a short observation about the two feature-target scatter plots.

## 8. Data Cleaning

In [ ]:
working_df = df.copy()

duplicate_count = working_df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count:,}")

if duplicate_count > 0:
    working_df = working_df.drop_duplicates().copy()
    print(f"Removed {duplicate_count:,} exact duplicate rows.")
else:
    print("No exact duplicate rows found.")

In [ ]:
missing_summary = (
    working_df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_percent=lambda x: x["missing_count"].div(len(working_df)).mul(100).round(2))
    .query("missing_count > 0")
    .sort_values("missing_percent", ascending=False)
)

display(missing_summary)

In [ ]:
outlier_rows = []
for column in eda_features:
    values = working_df[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = ((values < lower) | (values > upper)).sum()
    outlier_rows.append({
        "feature": column,
        "potential_outliers": int(outlier_count),
        "outlier_percent": round(outlier_count / len(values) * 100, 2),
    })

outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary)

Missing values are handled later by imputers fitted on training data only. Potential outliers are checked, but not automatically deleted, because extreme astronomical measurements can still be valid observations.

## 9. Feature Engineering

In [ ]:
if {"koi_depth", "koi_duration"}.issubset(working_df.columns):
    working_df["transit_depth_per_hour"] = (
        working_df["koi_depth"] / working_df["koi_duration"].replace(0, np.nan)
    )
    engineered_features = ["transit_depth_per_hour"]
else:
    engineered_features = []

print("Engineered features:", engineered_features)
if engineered_features:
    display(working_df[engineered_features].describe().T)

`transit_depth_per_hour` is calculated as `koi_depth / koi_duration`.

> STUDENT JUSTIFICATION TODO: Explain why this engineered feature may help model performance after reviewing the outputs.

## 10. Feature and Target Split

In [ ]:
rows_before = len(working_df)
modeling_df = working_df.dropna(subset=[target_column]).copy()
print(f"Rows removed because target is missing: {rows_before - len(modeling_df):,}")

columns_not_used = {
    target_column: "target column",
    "kepid": "identifier column",
    "koi_pdisposition": "another disposition label",
    "koi_score": "disposition score related to the target",
    "koi_fpflag_nt": "false-positive flag related to target decision",
    "koi_fpflag_ss": "false-positive flag related to target decision",
    "koi_fpflag_co": "false-positive flag related to target decision",
    "koi_fpflag_ec": "false-positive flag related to target decision",
}

columns_not_used = {
    column: reason
    for column, reason in columns_not_used.items()
    if column in modeling_df.columns
}

display(
    pd.DataFrame(
        [{"Column": column, "Reason": reason} for column, reason in columns_not_used.items()]
    )
)

In [ ]:
numeric_columns = modeling_df.select_dtypes(include=[np.number]).columns.tolist()
feature_columns = [column for column in numeric_columns if column not in columns_not_used]

all_missing_features = [column for column in feature_columns if modeling_df[column].isna().all()]
constant_features = [
    column for column in feature_columns
    if column not in all_missing_features and modeling_df[column].nunique(dropna=True) <= 1
]

feature_columns = [
    column for column in feature_columns
    if column not in set(all_missing_features + constant_features)
]

removed_feature_summary = pd.DataFrame([
    {"Column": column, "Reason": "all values are missing"}
    for column in all_missing_features
] + [
    {"Column": column, "Reason": "constant value"}
    for column in constant_features
])

if not removed_feature_summary.empty:
    display(removed_feature_summary.sort_values("Column"))

X = modeling_df[feature_columns].copy()
y = modeling_df[target_column].astype(str).copy()

numeric_features = feature_columns
categorical_features = []

print(f"Number of modelling features: {len(feature_columns)}")
print("Categorical features selected:", categorical_features)
print("Numerical features selected:")
print(numeric_features)

This notebook uses numeric modelling features only. Text labels, names, comments, links, and provenance columns are left out to keep preprocessing simple and explainable for Review 1.

## 11. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

display(pd.DataFrame({
    "train_percent": y_train.value_counts(normalize=True).mul(100).round(2),
    "test_percent": y_test.value_counts(normalize=True).mul(100).round(2),
}))

## 12. Preprocessing Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("Preprocessing will be fitted only inside model pipelines using X_train.")

Preprocessing follows the PDF rule: imputing and scaling are fitted on the training set only through the sklearn pipeline, then applied to the test set through the same fitted pipeline.

## 13. Gaussian Naive Bayes

Gaussian Naive Bayes applies Bayes' theorem with a Gaussian likelihood assumption for continuous features. It is simple and explainable, but correlated features can weaken the conditional independence assumption.

In [ ]:
def calculate_roc_auc(y_true, probabilities, classes):
    if len(classes) == 2:
        return roc_auc_score(y_true, probabilities[:, 1])

    y_true_binarized = label_binarize(y_true, classes=classes)
    return roc_auc_score(
        y_true_binarized,
        probabilities,
        multi_class="ovr",
        average="weighted",
    )


def evaluate_classifier(model_name, estimator, X_eval, y_eval):
    predictions = estimator.predict(X_eval)
    probabilities = estimator.predict_proba(X_eval)
    classes = estimator.classes_

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision_Weighted": precision_score(y_eval, predictions, average="weighted", zero_division=0),
        "Recall_Weighted": recall_score(y_eval, predictions, average="weighted", zero_division=0),
        "F1_Weighted": f1_score(y_eval, predictions, average="weighted", zero_division=0),
        "ROC_AUC_OvR": calculate_roc_auc(y_eval, probabilities, classes),
    }

    print(f"Classification report: {model_name}")
    print(classification_report(y_eval, predictions, labels=classes, zero_division=0))

    return metrics, predictions, probabilities


def show_confusion_matrix(y_true, predictions, classes, title):
    matrix = confusion_matrix(y_true, predictions, labels=classes)
    display_matrix = ConfusionMatrixDisplay(matrix, display_labels=classes)
    display_matrix.plot(cmap="Blues", xticks_rotation=25)
    plt.title(title)
    plt.xlabel("Predicted label")
    plt.ylabel("Actual label")
    plt.tight_layout()
    plt.show()

In [ ]:
gnb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", GaussianNB()),
])

gnb_pipeline.fit(X_train, y_train)

gnb_metrics, gnb_predictions, gnb_probabilities = evaluate_classifier(
    "Gaussian Naive Bayes",
    gnb_pipeline,
    X_test,
    y_test,
)

display(pd.DataFrame([gnb_metrics]).round(4))

## 14. Decision Tree Classifier

In [ ]:
tree_baseline_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

tree_baseline_pipeline.fit(X_train, y_train)

tree_baseline_metrics, tree_baseline_predictions, tree_baseline_probabilities = evaluate_classifier(
    "Decision Tree Classifier - Baseline",
    tree_baseline_pipeline,
    X_test,
    y_test,
)

display(pd.DataFrame([tree_baseline_metrics]).round(4))

## 15. Decision Tree Hyperparameter Tuning

In [ ]:
tree_tuning_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

param_grid = {
    "model__max_depth": [None, 3, 5, 8, 10],
}

tree_grid_search = GridSearchCV(
    tree_tuning_pipeline,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=5,
    n_jobs=-1,
)

tree_grid_search.fit(X_train, y_train)
tuned_tree_pipeline = tree_grid_search.best_estimator_

tuned_tree_metrics, tuned_tree_predictions, tuned_tree_probabilities = evaluate_classifier(
    "Decision Tree Classifier - Tuned",
    tuned_tree_pipeline,
    X_test,
    y_test,
)

display(pd.DataFrame([
    {"Item": "Best parameters", "Value": tree_grid_search.best_params_},
    {"Item": "Best CV weighted F1", "Value": round(tree_grid_search.best_score_, 4)},
    {"Item": "Baseline test weighted F1", "Value": round(tree_baseline_metrics["F1_Weighted"], 4)},
    {"Item": "Tuned test weighted F1", "Value": round(tuned_tree_metrics["F1_Weighted"], 4)},
]))

## 16. Evaluation Visualisations

In [ ]:
show_confusion_matrix(y_test, gnb_predictions, gnb_pipeline.classes_, "Gaussian Naive Bayes Confusion Matrix")
show_confusion_matrix(y_test, tree_baseline_predictions, tree_baseline_pipeline.classes_, "Decision Tree Baseline Confusion Matrix")
show_confusion_matrix(y_test, tuned_tree_predictions, tuned_tree_pipeline.classes_, "Decision Tree Tuned Confusion Matrix")

In [ ]:
classes = tuned_tree_pipeline.classes_
y_test_binarized = label_binarize(y_test, classes=classes)

plt.figure(figsize=(8, 6))
for class_index, class_name in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_binarized[:, class_index], tuned_tree_probabilities[:, class_index])
    plt.plot(fpr, tpr, label=class_name)

plt.plot([0, 1], [0, 1], "k--", label="Random baseline")
plt.title("Tuned Decision Tree OvR ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

## 17. Model Comparison

In [ ]:
comparison_table = pd.DataFrame([
    gnb_metrics,
    tree_baseline_metrics,
    tuned_tree_metrics,
]).round(4)

comparison_table = comparison_table.sort_values("F1_Weighted", ascending=False).reset_index(drop=True)
display(comparison_table)

> STUDENT OBSERVATION TODO: Compare models using the final table and confusion matrices.

## 18. Decision Tree Visualisation and Feature Importance

In [ ]:
fitted_preprocessor = tuned_tree_pipeline.named_steps["preprocess"]
fitted_tree = tuned_tree_pipeline.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

plt.figure(figsize=(24, 10))
plot_tree(
    fitted_tree,
    feature_names=feature_names,
    class_names=fitted_tree.classes_,
    max_depth=3,
    filled=True,
    rounded=True,
    impurity=False,
    fontsize=8,
)
plt.title("Tuned Decision Tree - Top Levels")
plt.tight_layout()
plt.show()

In [ ]:
feature_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": fitted_tree.feature_importances_,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(15))

plt.figure(figsize=(10, 7))
sns.barplot(data=feature_importance.head(15), y="Feature", x="Importance")
plt.title("Top Decision Tree Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

> STUDENT OBSERVATION TODO: Explain which features appear important after checking the table.

## 19. Remaining Classification Part-A Models

The course PDF lists five Part-A classifiers for Review 1:

- Logistic Regression
- K-Nearest Neighbors
- Gaussian Naive Bayes
- Decision Tree Classifier
- Support Vector Machine

Only Gaussian Naive Bayes and Decision Tree are implemented in this notebook version. The remaining models should use the same split and comparison-table format when added.